In [4]:

import torch
import torchvision
import pandas as pd
from pathlib import Path

D1_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-d1-original-rice-leaf-1701/"
    "Original Images"
)

SPLIT_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-verified-train-val-test-splits"
)

manifest_path = SPLIT_ROOT / "D1_split_manifest.csv"

assert D1_ROOT.is_dir()
assert manifest_path.is_file()

manifest = pd.read_csv(manifest_path)

assert len(manifest) == 1651
assert set(manifest["class_id"]) == set(range(8))

expected = {
    "train": 1156,
    "val": 247,
    "test": 248
}

actual = manifest["split"].value_counts().to_dict()

assert actual == expected

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Dataset:", "D1")
print("Total images:", len(manifest))
print("Classes:", manifest["class_id"].nunique())
print("Train:", actual["train"])
print("Validation:", actual["val"])
print("Test:", actual["test"])
print("\nREVISED NOTEBOOK SETUP: PASSED")


PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Dataset: D1
Total images: 1651
Classes: 8
Train: 1156
Validation: 247
Test: 248

REVISED NOTEBOOK SETUP: PASSED


In [5]:

import torch
import torch.nn as nn
from torchvision.models import squeezenet1_1

backbone = squeezenet1_1(weights=None)
backbone.eval()

print("=== FEATURE MODULES ===")

for index, module in backbone.features.named_children():
    print(index, type(module).__name__)

print("\n=== CONVOLUTION LAYERS ===")

conv_names = []

for name, module in backbone.features.named_modules():
    if isinstance(module, nn.Conv2d):
        conv_names.append(name)
        print(
            name,
            "Input:", module.in_channels,
            "Output:", module.out_channels,
            "Kernel:", module.kernel_size
        )

print("\n=== FEATURE TENSOR ===")

with torch.inference_mode():
    x = torch.zeros(1, 3, 227, 227)
    features = backbone.features(x)

print("Input:", tuple(x.shape))
print("Output:", tuple(features.shape))
print("Feature modules:", len(backbone.features))
print("Backbone convolution layers:", len(conv_names))

assert len(backbone.features) == 13
assert len(conv_names) == 25
assert features.shape == (1, 512, 14, 14)

print("\nBACKBONE STRUCTURE VERIFICATION: PASSED")


=== FEATURE MODULES ===
0 Conv2d
1 ReLU
2 MaxPool2d
3 Fire
4 Fire
5 MaxPool2d
6 Fire
7 Fire
8 MaxPool2d
9 Fire
10 Fire
11 Fire
12 Fire

=== CONVOLUTION LAYERS ===
0 Input: 3 Output: 64 Kernel: (3, 3)
3.squeeze Input: 64 Output: 16 Kernel: (1, 1)
3.expand1x1 Input: 16 Output: 64 Kernel: (1, 1)
3.expand3x3 Input: 16 Output: 64 Kernel: (3, 3)
4.squeeze Input: 128 Output: 16 Kernel: (1, 1)
4.expand1x1 Input: 16 Output: 64 Kernel: (1, 1)
4.expand3x3 Input: 16 Output: 64 Kernel: (3, 3)
6.squeeze Input: 128 Output: 32 Kernel: (1, 1)
6.expand1x1 Input: 32 Output: 128 Kernel: (1, 1)
6.expand3x3 Input: 32 Output: 128 Kernel: (3, 3)
7.squeeze Input: 256 Output: 32 Kernel: (1, 1)
7.expand1x1 Input: 32 Output: 128 Kernel: (1, 1)
7.expand3x3 Input: 32 Output: 128 Kernel: (3, 3)
9.squeeze Input: 256 Output: 48 Kernel: (1, 1)
9.expand1x1 Input: 48 Output: 192 Kernel: (1, 1)
9.expand3x3 Input: 48 Output: 192 Kernel: (3, 3)
10.squeeze Input: 384 Output: 48 Kernel: (1, 1)
10.expand1x1 Input: 48 Output: 1

In [6]:

import torch
import torch.nn as nn
from torchvision.models import squeezenet1_1

backbone = squeezenet1_1(weights=None)

groups = {
    "Initial convolution": [0],
    "Fire2": [3],
    "Fire3": [4],
    "Fire4": [6],
    "Fire5": [7],
    "Fire6": [9],
    "Fire7": [10],
    "Fire8": [11],
    "Fire9": [12]
}

total = 0

for group_name, indices in groups.items():
    parameters = []

    for index in indices:
        parameters.extend(
            list(backbone.features[index].parameters())
        )

    count = sum(p.numel() for p in parameters)
    total += count

    print(f"{group_name}: {count:,} parameters")

actual_total = sum(
    p.numel()
    for p in backbone.features.parameters()
)

print("\nGrouped feature parameters:", f"{total:,}")
print("Actual feature parameters:", f"{actual_total:,}")

assert total == actual_total
assert len(backbone.features) == 13

print("\nFEATURE PARAMETER MAPPING: PASSED")


Initial convolution: 1,792 parameters
Fire2: 11,408 parameters
Fire3: 12,432 parameters
Fire4: 45,344 parameters
Fire5: 49,440 parameters
Fire6: 104,880 parameters
Fire7: 111,024 parameters
Fire8: 188,992 parameters
Fire9: 197,184 parameters

Grouped feature parameters: 722,496
Actual feature parameters: 722,496

FEATURE PARAMETER MAPPING: PASSED


In [7]:

import torch
from torchvision.models import (
    squeezenet1_1,
    SqueezeNet1_1_Weights
)

backbone = squeezenet1_1(
    weights=SqueezeNet1_1_Weights.IMAGENET1K_V1
)

frozen_indices = (0, 3, 4)

for index in frozen_indices:
    for parameter in backbone.features[index].parameters():
        parameter.requires_grad = False

frozen_count = sum(
    p.numel()
    for p in backbone.features.parameters()
    if not p.requires_grad
)

trainable_count = sum(
    p.numel()
    for p in backbone.features.parameters()
    if p.requires_grad
)

total_count = sum(
    p.numel()
    for p in backbone.features.parameters()
)

assert frozen_count == 25632
assert trainable_count == 696864
assert total_count == 722496

assert all(
    not p.requires_grad
    for index in frozen_indices
    for p in backbone.features[index].parameters()
)

assert all(
    p.requires_grad
    for index in (6, 7, 9, 10, 11, 12)
    for p in backbone.features[index].parameters()
)

print("Frozen feature parameters:", f"{frozen_count:,}")
print("Trainable feature parameters:", f"{trainable_count:,}")
print("Total feature parameters:", f"{total_count:,}")

print("\nPAPER 6 LAYER FREEZING: PASSED")


Downloading: "https://download.pytorch.org/models/squeezenet1_1-b8a52dc0.pth" to /root/.cache/torch/hub/checkpoints/squeezenet1_1-b8a52dc0.pth


100%|██████████| 4.73M/4.73M [00:00<00:00, 66.8MB/s]

Frozen feature parameters: 25,632
Trainable feature parameters: 696,864
Total feature parameters: 722,496

PAPER 6 LAYER FREEZING: PASSED


In [8]:

import cv2
import numpy as np
from PIL import Image

def paper6_lab_minmax(image):
    rgb = np.asarray(
        image.convert("RGB"),
        dtype=np.float32
    ) / 255.0

    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB
    )

    lightness = lab[:, :, 0].copy()

    minimum = float(lightness.min())
    maximum = float(lightness.max())

    if maximum > minimum:
        lab[:, :, 0] = (
            100.0
            * (lightness - minimum)
            / (maximum - minimum)
        )

    enhanced_rgb = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    enhanced_rgb = np.rint(
        np.clip(enhanced_rgb, 0.0, 1.0) * 255.0
    ).astype(np.uint8)

    return Image.fromarray(enhanced_rgb)


row = manifest[
    manifest["split"] == "train"
].iloc[0]

image_path = D1_ROOT / str(row["rel_path"])

with Image.open(image_path) as image:
    resized = image.convert("RGB").resize(
        (227, 227),
        resample=Image.Resampling.BILINEAR
    )

enhanced_1 = paper6_lab_minmax(resized)
enhanced_2 = paper6_lab_minmax(resized)

array_1 = np.asarray(enhanced_1)
array_2 = np.asarray(enhanced_2)

assert enhanced_1.size == (227, 227)
assert enhanced_1.mode == "RGB"
assert array_1.dtype == np.uint8
assert np.array_equal(array_1, array_2)
assert np.isfinite(array_1).all()

constant_image = Image.new(
    "RGB",
    (227, 227),
    (70, 70, 70)
)

constant_result = paper6_lab_minmax(
    constant_image
)

assert constant_result.size == (227, 227)
assert np.isfinite(
    np.asarray(constant_result)
).all()

print("Original resized image:", resized.size)
print("Enhanced image:", enhanced_1.size)
print("Enhanced mode:", enhanced_1.mode)
print(
    "Output pixel range:",
    int(array_1.min()),
    int(array_1.max())
)
print(
    "Deterministic:",
    np.array_equal(array_1, array_2)
)
print("Constant-image safety: PASSED")
print("\nPAPER 6 CONTRAST PREPROCESSING: PASSED")


Original resized image: (227, 227)
Enhanced image: (227, 227)
Enhanced mode: RGB
Output pixel range: 0 255
Deterministic: True
Constant-image safety: PASSED

PAPER 6 CONTRAST PREPROCESSING: PASSED


In [9]:

import random
import cv2
import numpy as np
from PIL import Image

def paper6_augment(image, rng):
    array = np.array(image.convert("RGB"))
    height, width = array.shape[:2]

    flipped = rng.random() < 0.5

    if flipped:
        array = cv2.flip(array, 1)

    scale_x = rng.uniform(0.9, 1.1)
    scale_y = rng.uniform(0.9, 1.1)

    translate_x = rng.uniform(-30, 30)
    translate_y = rng.uniform(-30, 30)

    center_x = (width - 1) / 2
    center_y = (height - 1) / 2

    matrix = np.array([
        [
            scale_x,
            0.0,
            (1.0 - scale_x) * center_x + translate_x
        ],
        [
            0.0,
            scale_y,
            (1.0 - scale_y) * center_y + translate_y
        ]
    ], dtype=np.float32)

    augmented = cv2.warpAffine(
        array,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    parameters = {
        "horizontal_flip": flipped,
        "scale_x": scale_x,
        "scale_y": scale_y,
        "translate_x_px": translate_x,
        "translate_y_px": translate_y
    }

    return Image.fromarray(augmented), parameters


rng_1 = random.Random(2026)
rng_2 = random.Random(2026)

augmented_1, params_1 = paper6_augment(
    enhanced_1, rng_1
)

augmented_2, params_2 = paper6_augment(
    enhanced_1, rng_2
)

assert augmented_1.size == (227, 227)
assert augmented_1.mode == "RGB"
assert params_1 == params_2

assert np.array_equal(
    np.asarray(augmented_1),
    np.asarray(augmented_2)
)

assert 0.9 <= params_1["scale_x"] <= 1.1
assert 0.9 <= params_1["scale_y"] <= 1.1

assert -30 <= params_1["translate_x_px"] <= 30
assert -30 <= params_1["translate_y_px"] <= 30

assert np.isfinite(
    np.asarray(augmented_1)
).all()

print("Horizontal reflection:", params_1["horizontal_flip"])
print("Horizontal scale:", round(params_1["scale_x"], 4))
print("Vertical scale:", round(params_1["scale_y"], 4))
print("Horizontal translation:", round(params_1["translate_x_px"], 2))
print("Vertical translation:", round(params_1["translate_y_px"], 2))

print("Output size:", augmented_1.size)
print("Deterministic with same seed:", params_1 == params_2)

print("\nPAPER 6 AUGMENTATION VERIFICATION: PASSED")


Horizontal reflection: True
Horizontal scale: 1.0005
Vertical scale: 1.0024
Horizontal translation: 21.6
Vertical translation: -23.84
Output size: (227, 227)
Deterministic with same seed: True

PAPER 6 AUGMENTATION VERIFICATION: PASSED


In [10]:

import random
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class Paper6Transform:
    def __init__(self, training, seed=2026):
        self.training = training
        self.rng = random.Random(seed)
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __call__(self, image):
        image = image.convert("RGB").resize(
            (227, 227),
            Image.Resampling.BILINEAR
        )

        image = paper6_lab_minmax(image)

        if self.training:
            image, _ = paper6_augment(image, self.rng)

        return self.normalize(self.to_tensor(image))


class Paper6RiceDataset(Dataset):
    def __init__(self, rows, image_root, transform):
        self.rows = rows.reset_index(drop=True).copy()
        self.root = Path(image_root)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]

        with Image.open(
            self.root / str(row["rel_path"])
        ) as image:
            tensor = self.transform(image)

        return tensor, int(row["class_id"])


train_rows = manifest[
    manifest["split"] == "train"
].copy()

val_rows = manifest[
    manifest["split"] == "val"
].copy()

train_dataset = Paper6RiceDataset(
    train_rows,
    D1_ROOT,
    Paper6Transform(training=True, seed=2026)
)

val_dataset = Paper6RiceDataset(
    val_rows,
    D1_ROOT,
    Paper6Transform(training=False)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

train_images, train_labels = next(iter(train_loader))
val_images, val_labels = next(iter(val_loader))

val_first, _ = val_dataset[0]
val_second, _ = val_dataset[0]

assert len(train_dataset) == 1156
assert len(val_dataset) == 247

assert train_images.shape == (4, 3, 227, 227)
assert val_images.shape == (4, 3, 227, 227)

assert torch.isfinite(train_images).all()
assert torch.isfinite(val_images).all()

assert train_dataset.transform.training is True
assert val_dataset.transform.training is False

assert torch.equal(val_first, val_second)

assert set(train_rows["class_id"]) == set(range(8))
assert set(val_rows["class_id"]) == set(range(8))

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training batch:", tuple(train_images.shape))
print("Validation batch:", tuple(val_images.shape))
print("Validation deterministic:", torch.equal(val_first, val_second))
print("Training augmentation enabled:", train_dataset.transform.training)
print("Validation augmentation enabled:", val_dataset.transform.training)

print("\nPAPER 6 DATA PIPELINE: PASSED")


Training images: 1156
Validation images: 247
Training batch: (4, 3, 227, 227)
Validation batch: (4, 3, 227, 227)
Validation deterministic: True
Training augmentation enabled: True
Validation augmentation enabled: False

PAPER 6 DATA PIPELINE: PASSED


In [11]:

import torch
import torch.nn as nn

from torchvision.models import (
    squeezenet1_1,
    SqueezeNet1_1_Weights
)


class Paper6MFASqueezeNet(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        backbone = squeezenet1_1(
            weights=SqueezeNet1_1_Weights.IMAGENET1K_V1
        )

        self.features = backbone.features

        for index in (0, 3, 4):
            for parameter in self.features[index].parameters():
                parameter.requires_grad = False

        self.dropout = backbone.classifier[0]

        self.mfa_1x1 = nn.Conv2d(
            in_channels=512,
            out_channels=5,
            kernel_size=1
        )

        self.mfa_3x3 = nn.Conv2d(
            in_channels=512,
            out_channels=5,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU(inplace=False)

        self.class_projection = nn.Conv2d(
            in_channels=10,
            out_channels=num_classes,
            kernel_size=1
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        features = self.features(x)
        features = self.dropout(features)

        path1 = self.relu(self.mfa_1x1(features))
        path2 = self.relu(self.mfa_3x3(features))

        combined = torch.cat(
            (path1, path2),
            dim=1
        )

        class_maps = self.class_projection(combined)
        pooled = self.pool(class_maps)

        return torch.flatten(pooled, 1)


torch.manual_seed(2026)

model = Paper6MFASqueezeNet(num_classes=8)

model.eval()

x = torch.randn(1, 3, 227, 227)

with torch.inference_mode():
    features = model.features(x)

    path1 = model.mfa_1x1(features)
    path2 = model.mfa_3x3(features)

    combined = torch.cat(
        (path1, path2),
        dim=1
    )

    logits = model(x)

assert features.shape == (1, 512, 14, 14)
assert path1.shape == (1, 5, 14, 14)
assert path2.shape == (1, 5, 14, 14)
assert combined.shape == (1, 10, 14, 14)
assert logits.shape == (1, 8)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

frozen_parameters = sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 748194
assert frozen_parameters == 25632
assert trainable_parameters == 722562

model.train()
model.zero_grad(set_to_none=True)

training_logits = model(x)
target = torch.tensor([7])

loss = nn.CrossEntropyLoss()(
    training_logits,
    target
)

loss.backward()

assert torch.isfinite(loss)

assert model.features[0].weight.grad is None
assert model.features[3].squeeze.weight.grad is None
assert model.features[4].squeeze.weight.grad is None

assert model.features[6].squeeze.weight.grad is not None
assert model.mfa_1x1.weight.grad is not None
assert model.mfa_3x3.weight.grad is not None
assert model.class_projection.weight.grad is not None

assert all(
    p.grad is None or torch.isfinite(p.grad).all()
    for p in model.parameters()
)

print("Backbone features:", tuple(features.shape))
print("MFA 1x1 branch:", tuple(path1.shape))
print("MFA 3x3 branch:", tuple(path2.shape))
print("Concatenated features:", tuple(combined.shape))
print("Classification logits:", tuple(logits.shape))

print("\nTotal parameters:", f"{total_parameters:,}")
print("Frozen parameters:", f"{frozen_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

print("\nForward pass: PASSED")
print("Gradient verification: PASSED")
print("Layer freezing: PASSED")

print("\nPAPER 6 REVISED ARCHITECTURE: PASSED")


Backbone features: (1, 512, 14, 14)
MFA 1x1 branch: (1, 5, 14, 14)
MFA 3x3 branch: (1, 5, 14, 14)
Concatenated features: (1, 10, 14, 14)
Classification logits: (1, 8)

Total parameters: 748,194
Frozen parameters: 25,632
Trainable parameters: 722,562

Forward pass: PASSED
Gradient verification: PASSED
Layer freezing: PASSED

PAPER 6 REVISED ARCHITECTURE: PASSED


In [12]:

import torch

BASE_LR = 0.0003
NEW_LAYER_FACTOR = 10.0
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0001

backbone_weights = [
    p for name, p in model.features.named_parameters()
    if p.requires_grad and name.endswith("weight")
]

backbone_biases = [
    p for name, p in model.features.named_parameters()
    if p.requires_grad and name.endswith("bias")
]

parameter_groups = [
    {
        "params": backbone_weights,
        "lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "group_name": "backbone_weights"
    },
    {
        "params": backbone_biases,
        "lr": BASE_LR,
        "weight_decay": 0.0,
        "group_name": "backbone_biases"
    }
]

for layer_name in (
    "mfa_1x1",
    "mfa_3x3",
    "class_projection"
):
    layer = getattr(model, layer_name)

    parameter_groups.append({
        "params": [layer.weight],
        "lr": BASE_LR * NEW_LAYER_FACTOR,
        "weight_decay": WEIGHT_DECAY,
        "group_name": layer_name + "_weight"
    })

    parameter_groups.append({
        "params": [layer.bias],
        "lr": BASE_LR * NEW_LAYER_FACTOR,
        "weight_decay": 0.0,
        "group_name": layer_name + "_bias"
    })

optimizer = torch.optim.SGD(
    parameter_groups,
    lr=BASE_LR,
    momentum=MOMENTUM,
    nesterov=False
)

assigned = [
    p
    for group in optimizer.param_groups
    for p in group["params"]
]

trainable = [
    p for p in model.parameters()
    if p.requires_grad
]

assigned_ids = [id(p) for p in assigned]
trainable_ids = [id(p) for p in trainable]

assert len(optimizer.param_groups) == 8
assert len(assigned_ids) == len(set(assigned_ids))
assert set(assigned_ids) == set(trainable_ids)

assert sum(p.numel() for p in assigned) == 722562

assert sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
) == 25632

assert optimizer.defaults["momentum"] == MOMENTUM

for group in optimizer.param_groups:
    expected_lr = (
        BASE_LR
        if group["group_name"].startswith("backbone")
        else BASE_LR * NEW_LAYER_FACTOR
    )

    assert abs(group["lr"] - expected_lr) < 1e-12

    expected_decay = (
        0.0
        if group["group_name"].endswith("biases")
        or group["group_name"].endswith("_bias")
        else WEIGHT_DECAY
    )

    assert group["weight_decay"] == expected_decay

optimizer.zero_grad(set_to_none=True)

print("Optimizer:", type(optimizer).__name__)
print("Momentum:", optimizer.defaults["momentum"])
print("Parameter groups:", len(optimizer.param_groups))

for group in optimizer.param_groups:
    count = sum(p.numel() for p in group["params"])

    print(
        group["group_name"],
        "| Parameters:", count,
        "| LR:", round(group["lr"], 7),
        "| Weight decay:", group["weight_decay"]
    )

print(
    "\nOptimized parameters:",
    sum(p.numel() for p in assigned)
)

print("Duplicate parameters: None")
print("Frozen parameters excluded: Yes")
print("\nPAPER 6 OPTIMIZER CONFIGURATION: PASSED")


Optimizer: SGD
Momentum: 0.9
Parameter groups: 8
backbone_weights | Parameters: 694272 | LR: 0.0003 | Weight decay: 0.0001
backbone_biases | Parameters: 2592 | LR: 0.0003 | Weight decay: 0.0
mfa_1x1_weight | Parameters: 2560 | LR: 0.003 | Weight decay: 0.0001
mfa_1x1_bias | Parameters: 5 | LR: 0.003 | Weight decay: 0.0
mfa_3x3_weight | Parameters: 23040 | LR: 0.003 | Weight decay: 0.0001
mfa_3x3_bias | Parameters: 5 | LR: 0.003 | Weight decay: 0.0
class_projection_weight | Parameters: 80 | LR: 0.003 | Weight decay: 0.0001
class_projection_bias | Parameters: 8 | LR: 0.003 | Weight decay: 0.0

Optimized parameters: 722562
Duplicate parameters: None
Frozen parameters excluded: Yes

PAPER 6 OPTIMIZER CONFIGURATION: PASSED


In [13]:

import json
import hashlib
import random
import torch
import torchvision
import cv2
from pathlib import Path
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

SEED = 2026
BATCH_SIZE = 30

EXPECTED_SPLITS = {
    "train": 1156,
    "val": 247,
    "test": 248
}

assert manifest["split"].value_counts().to_dict() == EXPECTED_SPLITS
assert manifest["rel_path"].notna().all()
assert manifest["rel_path"].is_unique
assert manifest["class_id"].between(0, 7).all()

for split_name in EXPECTED_SPLITS:
    rows = manifest[manifest["split"] == split_name]
    assert set(rows["class_id"].astype(int)) == set(range(8))

missing = [
    str(D1_ROOT / str(path))
    for path in manifest["rel_path"]
    if not (D1_ROOT / str(path)).is_file()
]

assert not missing, f"Missing images: {missing[:5]}"

train_rows = manifest[
    manifest["split"] == "train"
].copy()

val_rows = manifest[
    manifest["split"] == "val"
].copy()

audit_train_dataset = Paper6RiceDataset(
    train_rows,
    D1_ROOT,
    Paper6Transform(training=True, seed=SEED)
)

audit_val_dataset = Paper6RiceDataset(
    val_rows,
    D1_ROOT,
    Paper6Transform(training=False)
)

audit_generator = torch.Generator().manual_seed(SEED)

audit_train_loader = DataLoader(
    audit_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=audit_generator,
    num_workers=0,
    drop_last=False
)

audit_val_loader = DataLoader(
    audit_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

assert len(audit_train_loader) == 39
assert len(audit_val_loader) == 9

assert isinstance(
    audit_train_loader.sampler,
    RandomSampler
)

assert isinstance(
    audit_val_loader.sampler,
    SequentialSampler
)

train_x, train_y = next(iter(audit_train_loader))
val_x, val_y = next(iter(audit_val_loader))

assert train_x.shape == (30, 3, 227, 227)
assert val_x.shape == (30, 3, 227, 227)

assert train_y.dtype == torch.int64
assert val_y.dtype == torch.int64

assert torch.isfinite(train_x).all()
assert torch.isfinite(val_x).all()

val_a, _ = audit_val_dataset[0]
val_b, _ = audit_val_dataset[0]

assert torch.equal(val_a, val_b)

assert audit_train_dataset.transform.training
assert not audit_val_dataset.transform.training

OUTPUT_DIR = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

configuration = {
    "paper": "Upadhyay and Dwivedi (2025), DOI: 10.1111/jph.70092",
    "implementation": "Independent PyTorch adaptation, not exact MATLAB reproduction",
    "dataset": "D1, eight classes",
    "manifest_sha256": hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest(),
    "split_counts": EXPECTED_SPLITS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": 10,
    "input_size": [227, 227],
    "backbone": "ImageNet-pretrained SqueezeNet 1.1",
    "frozen_feature_indices": [0, 3, 4],
    "mfa_branch_filters": [5, 5],
    "class_projection": "10-to-8 pointwise convolution",
    "preprocessing": "RGB resize, Lab L* min-max stretch, ImageNet normalization",
    "augmentation": "Horizontal reflection, +/-30-pixel translations, independent 0.9-1.1 scaling",
    "optimizer": {
        "name": "SGD",
        "backbone_lr": 0.0003,
        "new_layer_lr": 0.003,
        "momentum": 0.9,
        "weight_decay_weights": 0.0001,
        "weight_decay_biases": 0.0
    },
    "software": {
        "pytorch": torch.__version__,
        "torchvision": torchvision.__version__,
        "opencv": cv2.__version__
    },
    "disclosures": [
        "The paper does not specify the MFA class-projection operation.",
        "The original paper used five classes, a 70/30 split, and six epochs.",
        "D1 uses fixed duplicate-aware train, validation, and test partitions.",
        "New-layer learning-rate factor and several defaults are reconstruction choices.",
        "The D1 test set was previously evaluated in an exploratory run."
    ]
}

config_path = OUTPUT_DIR / "pretraining_config.json"

config_path.write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8"
)

print("Train:", len(audit_train_dataset))
print("Validation:", len(audit_val_dataset))
print("Test:", EXPECTED_SPLITS["test"])

print("Train batches:", len(audit_train_loader))
print("Validation batches:", len(audit_val_loader))

print("Training batch:", tuple(train_x.shape))
print("Validation batch:", tuple(val_x.shape))

print(
    "Train shuffle:",
    isinstance(audit_train_loader.sampler, RandomSampler)
)

print(
    "Validation deterministic:",
    torch.equal(val_a, val_b)
)

print("Missing image files:", len(missing))
print("Saved configuration:", config_path)

print("\nTRAINING READINESS AUDIT: PASSED")


Train: 1156
Validation: 247
Test: 248
Train batches: 39
Validation batches: 9
Training batch: (30, 3, 227, 227)
Validation batch: (30, 3, 227, 227)
Train shuffle: True
Validation deterministic: True
Missing image files: 0
Saved configuration: /kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/pretraining_config.json

TRAINING READINESS AUDIT: PASSED


In [14]:

import json
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "Enable Kaggle GPU first."

device = torch.device("cuda:0")

torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def build_paper6_optimizer(net):
    groups = [
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("weight")
            ],
            "lr": 0.0003,
            "weight_decay": 0.0001
        },
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("bias")
            ],
            "lr": 0.0003,
            "weight_decay": 0.0
        }
    ]

    for name in (
        "mfa_1x1",
        "mfa_3x3",
        "class_projection"
    ):
        layer = getattr(net, name)

        groups.append({
            "params": [layer.weight],
            "lr": 0.003,
            "weight_decay": 0.0001
        })

        groups.append({
            "params": [layer.bias],
            "lr": 0.003,
            "weight_decay": 0.0
        })

    opt = torch.optim.SGD(
        groups,
        momentum=0.9,
        nesterov=False
    )

    assigned = [
        p for group in opt.param_groups
        for p in group["params"]
    ]

    trainable = [
        p for p in net.parameters()
        if p.requires_grad
    ]

    assert len(opt.param_groups) == 8
    assert len(assigned) == len(set(map(id, assigned)))
    assert set(map(id, assigned)) == set(map(id, trainable))
    assert sum(p.numel() for p in assigned) == 722562

    return opt


smoke_model = Paper6MFASqueezeNet(
    num_classes=8
).to(device)

smoke_optimizer = build_paper6_optimizer(
    smoke_model
)

smoke_model.train()

input_x = train_x.to(device)
input_y = train_y.to(device)

assert input_x.shape == (30, 3, 227, 227)
assert input_y.shape == (30,)

frozen_before = (
    smoke_model.features[0].weight
    .detach().clone()
)

head_before = (
    smoke_model.class_projection.weight
    .detach().clone()
)

smoke_optimizer.zero_grad(set_to_none=True)

logits = smoke_model(input_x)
loss = nn.CrossEntropyLoss()(logits, input_y)

assert logits.shape == (30, 8)
assert torch.isfinite(loss)

loss.backward()

assert smoke_model.features[0].weight.grad is None
assert smoke_model.class_projection.weight.grad is not None

assert all(
    p.grad is None or torch.isfinite(p.grad).all()
    for p in smoke_model.parameters()
)

smoke_optimizer.step()

assert torch.equal(
    frozen_before,
    smoke_model.features[0].weight.detach()
)

assert not torch.equal(
    head_before,
    smoke_model.class_projection.weight.detach()
)

smoke_model.eval()

with torch.inference_mode():
    validation_logits = smoke_model(
        val_x.to(device)
    )

assert validation_logits.shape == (30, 8)
assert torch.isfinite(validation_logits).all()

checkpoint_path = OUTPUT_DIR / "gpu_smoke_checkpoint.pth"

torch.save(
    {
        "epoch": 0,
        "model_state_dict": smoke_model.state_dict(),
        "optimizer_state_dict": smoke_optimizer.state_dict()
    },
    checkpoint_path
)

restored = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert restored["epoch"] == 0

assert torch.equal(
    restored["model_state_dict"]["class_projection.weight"],
    smoke_model.class_projection.weight.detach().cpu()
)

checkpoint_path.unlink()

report = {
    "status": "PASSED",
    "purpose": "GPU smoke test only, not full training",
    "gpu": torch.cuda.get_device_name(0),
    "batch_size": int(input_x.shape[0]),
    "training_loss": float(loss.detach().cpu().item()),
    "frozen_weights_unchanged": True,
    "classifier_weights_updated": True,
    "validation_forward_passed": True,
    "checkpoint_roundtrip_passed": True
}

report_path = OUTPUT_DIR / "gpu_smoke_verification.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("GPU:", report["gpu"])
print("Training batch:", tuple(input_x.shape))
print("Training loss:", round(report["training_loss"], 6))
print("Frozen weights unchanged: True")
print("Classifier weights updated: True")
print("Validation forward pass: PASSED")
print("Checkpoint save/load: PASSED")
print("Verification report:", report_path)

print("\nPAPER 6 GPU SMOKE TEST: PASSED")


GPU: Tesla T4
Training batch: (30, 3, 227, 227)
Training loss: 2.736561
Frozen weights unchanged: True
Classifier weights updated: True
Validation forward pass: PASSED
Checkpoint save/load: PASSED
Verification report: /kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/gpu_smoke_verification.json

PAPER 6 GPU SMOKE TEST: PASSED


In [15]:

import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU VERIFICATION: PASSED")
else:
    print("GPU VERIFICATION: FAILED")
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(
        result.stdout or result.stderr
        or "NVIDIA GPU is not visible in this session."
    )


PyTorch version: 2.11.0+cu130
PyTorch CUDA build: 13.0
CUDA available: True
CUDA device count: 2
GPU: Tesla T4
GPU VERIFICATION: PASSED


In [16]:

import json
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "Enable Kaggle GPU first."

device = torch.device("cuda:0")

torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def build_paper6_optimizer(net):
    groups = [
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("weight")
            ],
            "lr": 0.0003,
            "weight_decay": 0.0001
        },
        {
            "params": [
                p for n, p in net.features.named_parameters()
                if p.requires_grad and n.endswith("bias")
            ],
            "lr": 0.0003,
            "weight_decay": 0.0
        }
    ]

    for name in (
        "mfa_1x1",
        "mfa_3x3",
        "class_projection"
    ):
        layer = getattr(net, name)

        groups.append({
            "params": [layer.weight],
            "lr": 0.003,
            "weight_decay": 0.0001
        })

        groups.append({
            "params": [layer.bias],
            "lr": 0.003,
            "weight_decay": 0.0
        })

    opt = torch.optim.SGD(
        groups,
        momentum=0.9,
        nesterov=False
    )

    assigned = [
        p for group in opt.param_groups
        for p in group["params"]
    ]

    trainable = [
        p for p in net.parameters()
        if p.requires_grad
    ]

    assert len(opt.param_groups) == 8
    assert len(assigned) == len(set(map(id, assigned)))
    assert set(map(id, assigned)) == set(map(id, trainable))
    assert sum(p.numel() for p in assigned) == 722562

    return opt


smoke_model = Paper6MFASqueezeNet(
    num_classes=8
).to(device)

smoke_optimizer = build_paper6_optimizer(
    smoke_model
)

smoke_model.train()

input_x = train_x.to(device)
input_y = train_y.to(device)

assert input_x.shape == (30, 3, 227, 227)
assert input_y.shape == (30,)

frozen_before = (
    smoke_model.features[0].weight
    .detach().clone()
)

head_before = (
    smoke_model.class_projection.weight
    .detach().clone()
)

smoke_optimizer.zero_grad(set_to_none=True)

logits = smoke_model(input_x)
loss = nn.CrossEntropyLoss()(logits, input_y)

assert logits.shape == (30, 8)
assert torch.isfinite(loss)

loss.backward()

assert smoke_model.features[0].weight.grad is None
assert smoke_model.class_projection.weight.grad is not None

assert all(
    p.grad is None or torch.isfinite(p.grad).all()
    for p in smoke_model.parameters()
)

smoke_optimizer.step()

assert torch.equal(
    frozen_before,
    smoke_model.features[0].weight.detach()
)

assert not torch.equal(
    head_before,
    smoke_model.class_projection.weight.detach()
)

smoke_model.eval()

with torch.inference_mode():
    validation_logits = smoke_model(
        val_x.to(device)
    )

assert validation_logits.shape == (30, 8)
assert torch.isfinite(validation_logits).all()

checkpoint_path = OUTPUT_DIR / "gpu_smoke_checkpoint.pth"

torch.save(
    {
        "epoch": 0,
        "model_state_dict": smoke_model.state_dict(),
        "optimizer_state_dict": smoke_optimizer.state_dict()
    },
    checkpoint_path
)

restored = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert restored["epoch"] == 0

assert torch.equal(
    restored["model_state_dict"]["class_projection.weight"],
    smoke_model.class_projection.weight.detach().cpu()
)

checkpoint_path.unlink()

report = {
    "status": "PASSED",
    "purpose": "GPU smoke test only, not full training",
    "gpu": torch.cuda.get_device_name(0),
    "batch_size": int(input_x.shape[0]),
    "training_loss": float(loss.detach().cpu().item()),
    "frozen_weights_unchanged": True,
    "classifier_weights_updated": True,
    "validation_forward_passed": True,
    "checkpoint_roundtrip_passed": True
}

report_path = OUTPUT_DIR / "gpu_smoke_verification.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("GPU:", report["gpu"])
print("Training batch:", tuple(input_x.shape))
print("Training loss:", round(report["training_loss"], 6))
print("Frozen weights unchanged: True")
print("Classifier weights updated: True")
print("Validation forward pass: PASSED")
print("Checkpoint save/load: PASSED")
print("Verification report:", report_path)

print("\nPAPER 6 GPU SMOKE TEST: PASSED")


GPU: Tesla T4
Training batch: (30, 3, 227, 227)
Training loss: 2.736561
Frozen weights unchanged: True
Classifier weights updated: True
Validation forward pass: PASSED
Checkpoint save/load: PASSED
Verification report: /kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/gpu_smoke_verification.json

PAPER 6 GPU SMOKE TEST: PASSED


In [17]:

import csv
import hashlib
import json
import os
import random
import time
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

assert torch.cuda.is_available(), "Kaggle GPU is required."

SEED = 2026
EPOCHS = 10
BATCH_SIZE = 30
DEVICE = torch.device("cuda:0")

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

config_file = output_dir / "pretraining_config.json"
last_file = output_dir / "last_checkpoint.pth"
best_file = output_dir / "best_model.pth"
history_file = output_dir / "training_history.csv"
archive_file = output_dir / "training_backup.zip"

config = json.loads(
    config_file.read_text(encoding="utf-8")
)

manifest_hash = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

assert config["manifest_sha256"] == manifest_hash
assert config["epochs"] == EPOCHS
assert config["batch_size"] == BATCH_SIZE
assert config["optimizer"]["backbone_lr"] == 0.0003
assert config["optimizer"]["new_layer_lr"] == 0.003

assert manifest["split"].value_counts().to_dict() == {
    "train": 1156,
    "val": 247,
    "test": 248
}

if not last_file.exists() and (
    history_file.exists() or best_file.exists()
):
    raise RuntimeError(
        "Existing training artifacts found without a "
        "resumable checkpoint. Preserve them first."
    )

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

training_rows = manifest.loc[
    manifest["split"] == "train"
].copy()

validation_rows = manifest.loc[
    manifest["split"] == "val"
].copy()

training_dataset = Paper6RiceDataset(
    training_rows,
    D1_ROOT,
    Paper6Transform(training=True, seed=SEED)
)

validation_dataset = Paper6RiceDataset(
    validation_rows,
    D1_ROOT,
    Paper6Transform(training=False)
)

shuffle_generator = torch.Generator().manual_seed(SEED)

training_loader = DataLoader(
    training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=shuffle_generator,
    num_workers=0,
    drop_last=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

assert len(training_loader) == 39
assert len(validation_loader) == 9

training_model = Paper6MFASqueezeNet(
    num_classes=8
).to(DEVICE)

training_optimizer = build_paper6_optimizer(
    training_model
)

criterion = nn.CrossEntropyLoss()

assert sum(
    p.numel()
    for p in training_model.parameters()
    if p.requires_grad
) == 722562

assert sum(
    p.numel()
    for p in training_model.parameters()
    if not p.requires_grad
) == 25632


def atomic_torch_save(value, path):
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)


def atomic_history_save(rows, path):
    temporary = path.with_name(path.name + ".tmp")

    with temporary.open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "epoch",
                "train_loss",
                "train_accuracy",
                "val_loss",
                "val_accuracy",
                "seconds"
            ]
        )
        writer.writeheader()
        writer.writerows(rows)

    os.replace(temporary, path)


def atomic_backup(path):
    temporary = path.with_name(path.name + ".tmp")

    with zipfile.ZipFile(
        temporary,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=3
    ) as archive:
        for item in (
            config_file,
            history_file,
            last_file,
            best_file
        ):
            if item.exists():
                archive.write(
                    item,
                    arcname=item.name
                )

    os.replace(temporary, path)


start_epoch = 1
best_loss = float("inf")
best_epoch = None
history = []

if last_file.exists():
    checkpoint = torch.load(
        last_file,
        map_location=DEVICE,
        weights_only=True
    )

    assert checkpoint["manifest_sha256"] == manifest_hash
    assert checkpoint["seed"] == SEED
    assert checkpoint["batch_size"] == BATCH_SIZE

    training_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    training_optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    shuffle_generator.set_state(
        checkpoint["shuffle_generator_state"]
    )

    training_dataset.transform.rng.setstate(
        checkpoint["augmentation_rng_state"]
    )

    torch.set_rng_state(
        checkpoint["torch_cpu_rng_state"].cpu()
    )

    torch.cuda.set_rng_state_all(
        checkpoint["torch_cuda_rng_states"]
    )

    history = checkpoint["history"]
    best_loss = float(checkpoint["best_val_loss"])
    best_epoch = int(checkpoint["best_epoch"])
    start_epoch = int(checkpoint["epoch"]) + 1

    assert len(history) == start_epoch - 1
    assert best_file.exists()

    print(
        "RESUMING FROM COMPLETED EPOCH:",
        start_epoch - 1,
        flush=True
    )

else:
    print(
        "STARTING FRESH TRAINING FROM PRETRAINED IMAGENET WEIGHTS",
        flush=True
    )

print("GPU:", torch.cuda.get_device_name(0), flush=True)
print(
    "D1 training/validation:",
    len(training_dataset),
    len(validation_dataset),
    flush=True
)
print("Test set: NOT USED", flush=True)


for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.perf_counter()

    training_model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, targets in training_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        training_optimizer.zero_grad(
            set_to_none=True
        )

        outputs = training_model(images)
        loss = criterion(outputs, targets)

        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Nonfinite training loss at epoch {epoch}."
            )

        loss.backward()
        training_optimizer.step()

        count = targets.size(0)

        train_loss_sum += (
            float(loss.detach().item()) * count
        )

        train_correct += int(
            (
                outputs.detach().argmax(dim=1)
                == targets
            ).sum().item()
        )

        train_count += count

    training_model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0

    with torch.inference_mode():
        for images, targets in validation_loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            outputs = training_model(images)
            loss = criterion(outputs, targets)

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"Nonfinite validation loss at epoch {epoch}."
                )

            count = targets.size(0)

            val_loss_sum += (
                float(loss.item()) * count
            )

            val_correct += int(
                (
                    outputs.argmax(dim=1)
                    == targets
                ).sum().item()
            )

            val_count += count

    assert train_count == 1156
    assert val_count == 247

    train_loss = train_loss_sum / train_count
    val_loss = val_loss_sum / val_count

    train_accuracy = train_correct / train_count
    val_accuracy = val_correct / val_count

    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "seconds": elapsed
    })

    if val_loss < best_loss:
        best_loss = val_loss
        best_epoch = epoch

        atomic_torch_save(
            {
                "epoch": epoch,
                "model_state_dict": training_model.state_dict(),
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
                "manifest_sha256": manifest_hash,
                "num_classes": 8
            },
            best_file
        )

    atomic_torch_save(
        {
            "epoch": epoch,
            "model_state_dict": training_model.state_dict(),
            "optimizer_state_dict": training_optimizer.state_dict(),
            "best_epoch": best_epoch,
            "best_val_loss": best_loss,
            "history": history,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "manifest_sha256": manifest_hash,
            "shuffle_generator_state": shuffle_generator.get_state(),
            "augmentation_rng_state": training_dataset.transform.rng.getstate(),
            "torch_cpu_rng_state": torch.get_rng_state(),
            "torch_cuda_rng_states": torch.cuda.get_rng_state_all()
        },
        last_file
    )

    atomic_history_save(
        history,
        history_file
    )

    atomic_backup(
        archive_file
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_loss:.4f}, "
        f"acc {train_accuracy:.4f} | "
        f"Val loss {val_loss:.4f}, "
        f"acc {val_accuracy:.4f} | "
        f"{elapsed:.1f}s | "
        f"best epoch {best_epoch} | saved",
        flush=True
    )

print("\nCOMPLETED EPOCHS:", len(history))
print("BEST VALIDATION EPOCH:", best_epoch)
print("BEST VALIDATION LOSS:", round(best_loss, 6))
print("HISTORY:", history_file)
print("BEST MODEL:", best_file)
print("RESUME CHECKPOINT:", last_file)
print("BACKUP ARCHIVE:", archive_file)

print("\nPAPER 6 D1 REVISED TRAINING: COMPLETE")


STARTING FRESH TRAINING FROM PRETRAINED IMAGENET WEIGHTS
GPU: Tesla T4
D1 training/validation: 1156 247
Test set: NOT USED
Epoch 01/10 | Train loss 2.0777, acc 0.2050 | Val loss 1.9540, acc 0.2510 | 191.3s | best epoch 1 | saved
Epoch 02/10 | Train loss 1.8243, acc 0.3071 | Val loss 1.7297, acc 0.3239 | 56.5s | best epoch 2 | saved
Epoch 03/10 | Train loss 1.4856, acc 0.4533 | Val loss 1.3694, acc 0.4818 | 56.9s | best epoch 3 | saved
Epoch 04/10 | Train loss 1.1459, acc 0.5640 | Val loss 1.0813, acc 0.5951 | 56.0s | best epoch 4 | saved
Epoch 05/10 | Train loss 0.9556, acc 0.6479 | Val loss 1.0069, acc 0.6194 | 55.9s | best epoch 5 | saved
Epoch 06/10 | Train loss 0.9097, acc 0.6721 | Val loss 1.0191, acc 0.6275 | 56.1s | best epoch 5 | saved
Epoch 07/10 | Train loss 0.8138, acc 0.6955 | Val loss 0.7836, acc 0.6964 | 55.9s | best epoch 7 | saved
Epoch 08/10 | Train loss 0.7832, acc 0.7163 | Val loss 0.9114, acc 0.6842 | 56.4s | best epoch 7 | saved
Epoch 09/10 | Train loss 0.6995, acc

In [18]:

import csv
import math
import zipfile
import hashlib
import torch
from pathlib import Path
from IPython.display import FileLink, display

folder = Path(
    "/kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised"
)

required = [
    "pretraining_config.json",
    "training_history.csv",
    "best_model.pth",
    "last_checkpoint.pth",
    "training_backup.zip",
    "gpu_smoke_verification.json"
]

for name in required:
    path = folder / name
    assert path.is_file(), f"Missing file: {name}"
    assert path.stat().st_size > 0, f"Empty file: {name}"

with (folder / "training_history.csv").open(
    newline="", encoding="utf-8"
) as handle:
    rows = list(csv.DictReader(handle))

assert len(rows) == 10
assert [int(row["epoch"]) for row in rows] == list(range(1, 11))

best_row = min(
    rows,
    key=lambda row: float(row["val_loss"])
)

assert int(best_row["epoch"]) == 10
assert math.isclose(
    float(best_row["val_loss"]),
    0.736064,
    abs_tol=0.00001
)

best_checkpoint = torch.load(
    folder / "best_model.pth",
    map_location="cpu",
    weights_only=True
)

last_checkpoint = torch.load(
    folder / "last_checkpoint.pth",
    map_location="cpu",
    weights_only=True
)

assert best_checkpoint["epoch"] == 10
assert last_checkpoint["epoch"] == 10
assert last_checkpoint["best_epoch"] == 10

archive_path = folder / "training_backup.zip"

with zipfile.ZipFile(archive_path) as archive:
    names = set(archive.namelist())

    assert {
        "pretraining_config.json",
        "training_history.csv",
        "best_model.pth",
        "last_checkpoint.pth"
    }.issubset(names)

    assert archive.testzip() is None

archive_hash = hashlib.sha256(
    archive_path.read_bytes()
).hexdigest()

print("Training epochs:", len(rows))
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation loss:", best_row["val_loss"])
print("Last checkpoint epoch:", last_checkpoint["epoch"])
print("Backup integrity: PASSED")
print("Archive SHA256:", archive_hash)
print("Archive size (MB):", round(
    archive_path.stat().st_size / 1048576, 2
))
print("\nPAPER 6 TRAINING ARTIFACT AUDIT: PASSED")

display(FileLink(str(archive_path)))


Training epochs: 10
Best epoch: 10
Best validation loss: 0.7360640367274343
Last checkpoint epoch: 10
Backup integrity: PASSED
Archive SHA256: 6e4a6cda8b4dde1aaf65f4f1efffe6ef5b89789a47764f652680d79e1c3a10df
Archive size (MB): 7.93

PAPER 6 TRAINING ARTIFACT AUDIT: PASSED


/kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/training_backup.zip

In [19]:

import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

result_dir = Path(OUTPUT_DIR) / "evaluation"
result_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = Path(OUTPUT_DIR) / "best_model.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["epoch"] == 10
assert checkpoint["num_classes"] == 8

current_manifest_hash = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

assert checkpoint["manifest_sha256"] == current_manifest_hash

test_rows = manifest.loc[
    manifest["split"] == "test"
].reset_index(drop=True)

assert len(test_rows) == 248
assert set(test_rows["class_id"].astype(int)) == set(range(8))

test_dataset = Paper6RiceDataset(
    test_rows,
    D1_ROOT,
    Paper6Transform(training=False)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=30,
    shuffle=False,
    num_workers=0
)

evaluation_model = Paper6MFASqueezeNet(
    num_classes=8
)

evaluation_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

evaluation_model = evaluation_model.to(device)
evaluation_model.eval()

true_labels = []
predicted_labels = []
probability_batches = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)

        logits = evaluation_model(images)

        assert logits.ndim == 2
        assert logits.shape[1] == 8
        assert torch.isfinite(logits).all()

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(dim=1)

        true_labels.extend(labels.numpy().tolist())

        predicted_labels.extend(
            predictions.cpu().numpy().tolist()
        )

        probability_batches.append(
            probabilities.cpu().numpy()
        )

y_true = np.asarray(true_labels, dtype=int)
y_pred = np.asarray(predicted_labels, dtype=int)

y_prob = np.concatenate(
    probability_batches,
    axis=0
)

class_ids = np.arange(8)

assert y_true.shape == (248,)
assert y_pred.shape == (248,)
assert y_prob.shape == (248, 8)

assert np.isfinite(y_prob).all()
assert np.allclose(y_prob.sum(axis=1), 1.0, atol=1e-6)
assert set(y_true.tolist()) == set(class_ids.tolist())
assert set(y_pred.tolist()).issubset(set(class_ids.tolist()))

accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=class_ids,
        zero_division=0
    )
)

macro_precision = float(np.mean(precision))
macro_recall = float(np.mean(recall))
macro_f1 = float(np.mean(f1))

auc_macro = roc_auc_score(
    y_true,
    y_prob,
    labels=class_ids,
    multi_class="ovr",
    average="macro"
)

auc_weighted = roc_auc_score(
    y_true,
    y_prob,
    labels=class_ids,
    multi_class="ovr",
    average="weighted"
)

class_auc = []

for class_id in class_ids:
    binary_truth = (y_true == class_id).astype(int)

    class_auc.append(
        roc_auc_score(
            binary_truth,
            y_prob[:, class_id]
        )
    )

metrics = {
    "experiment": "Paper6_MFA_SqueezeNet_D1_Revised",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_samples": len(y_true),
    "num_classes": 8,
    "accuracy": float(accuracy),
    "precision_macro": macro_precision,
    "recall_macro": macro_recall,
    "f1_macro": macro_f1,
    "auc_macro_ovr": float(auc_macro),
    "auc_weighted_ovr": float(auc_weighted),
    "test_set_disclosure": (
        "The D1 test set was previously evaluated "
        "during exploratory implementation. "
        "This revised evaluation is not a pristine holdout."
    )
}

metrics_path = result_dir / "test_metrics.json"

metrics_path.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

classwise = pd.DataFrame({
    "class_id": class_ids,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support,
    "auc_ovr": class_auc
})

classwise.to_csv(
    result_dir / "classwise_metrics.csv",
    index=False
)

confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=class_ids
)

confusion_df = pd.DataFrame(
    confusion,
    index=[f"actual_{i}" for i in class_ids],
    columns=[f"predicted_{i}" for i in class_ids]
)

confusion_df.to_csv(
    result_dir / "confusion_matrix.csv"
)

predictions_df = pd.DataFrame({
    "rel_path": test_rows["rel_path"].astype(str),
    "true_class_id": y_true,
    "predicted_class_id": y_pred,
    "correct": y_true == y_pred
})

for class_id in class_ids:
    predictions_df[f"probability_{class_id}"] = (
        y_prob[:, class_id]
    )

predictions_df.to_csv(
    result_dir / "test_predictions.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(8, 7))

ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=class_ids
).plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("Paper 6 Revised D1 - Test Confusion Matrix")
fig.tight_layout()

fig.savefig(
    result_dir / "confusion_matrix.png",
    dpi=180
)

plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))

for class_id in class_ids:
    binary_truth = (y_true == class_id).astype(int)

    false_positive_rate, true_positive_rate, _ = (
        roc_curve(
            binary_truth,
            y_prob[:, class_id]
        )
    )

    ax.plot(
        false_positive_rate,
        true_positive_rate,
        label=f"Class {class_id} (AUC={class_auc[class_id]:.3f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Paper 6 Revised D1 - One-vs-Rest ROC")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

fig.tight_layout()

fig.savefig(
    result_dir / "roc_curves.png",
    dpi=180
)

plt.close(fig)

archive_path = result_dir / "evaluation_results.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in (
        "test_metrics.json",
        "classwise_metrics.csv",
        "confusion_matrix.csv",
        "confusion_matrix.png",
        "roc_curves.png",
        "test_predictions.csv"
    ):
        archive.write(
            result_dir / filename,
            arcname=filename
        )

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None

assert sum(support) == 248
assert confusion.sum() == 248
assert int(np.trace(confusion)) == int(
    np.sum(y_true == y_pred)
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Test images:", len(y_true))
print("Accuracy:", round(accuracy, 6))
print("Macro precision:", round(macro_precision, 6))
print("Macro recall:", round(macro_recall, 6))
print("Macro F1:", round(macro_f1, 6))
print("Macro AUC (OvR):", round(auc_macro, 6))
print("Weighted AUC (OvR):", round(auc_weighted, 6))

print("\nCLASS-WISE METRICS")
print(classwise.to_string(index=False))

print("\nCONFUSION MATRIX")
print(confusion)

print("\nResults directory:", result_dir)
print("Evaluation archive:", archive_path)

print("\nPAPER 6 D1 TEST EVALUATION: PASSED")


Checkpoint epoch: 10
Test images: 248
Accuracy: 0.697581
Macro precision: 0.659741
Macro recall: 0.662813
Macro F1: 0.64824
Macro AUC (OvR): 0.933551
Weighted AUC (OvR): 0.939996

CLASS-WISE METRICS
 class_id  precision   recall  f1_score  support  auc_ovr
        0   0.625000 0.740741  0.677966       27 0.939836
        1   0.769231 0.769231  0.769231       39 0.952398
        2   0.882353 0.652174  0.750000       23 0.995749
        3   0.536585 0.536585  0.536585       41 0.890185
        4   0.578947 0.785714  0.666667       28 0.947078
        5   0.250000 0.055556  0.090909       18 0.776812
        6   0.810811 0.937500  0.869565       32 0.994358
        7   0.825000 0.825000  0.825000       40 0.971995

CONFUSION MATRIX
[[20  0  0  2  4  0  0  1]
 [ 0 30  0  6  2  0  0  1]
 [ 0  4 15  0  0  0  4  0]
 [ 5  0  0 22  5  3  2  4]
 [ 4  0  0  1 22  0  0  1]
 [ 2  5  0  7  3  1  0  0]
 [ 0  0  1  1  0  0 30  0]
 [ 1  0  1  2  2  0  1 33]]

Results directory: /kaggle/working/Paper6_M

In [20]:

import json
import math
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import FileLink, display

root = Path(OUTPUT_DIR)
evaluation_dir = root / "evaluation"

class_map_path = SPLIT_ROOT / "class_maps.json"

assert class_map_path.is_file()

class_maps = json.loads(
    class_map_path.read_text(encoding="utf-8")
)

metrics = json.loads(
    (evaluation_dir / "test_metrics.json").read_text(
        encoding="utf-8"
    )
)

classwise = pd.read_csv(
    evaluation_dir / "classwise_metrics.csv"
)

confusion = pd.read_csv(
    evaluation_dir / "confusion_matrix.csv",
    index_col=0
).to_numpy(dtype=int)

assert confusion.shape == (8, 8)
assert confusion.sum() == 248
assert np.trace(confusion) == 173

assert list(classwise["class_id"]) == list(range(8))
assert np.array_equal(
    confusion.sum(axis=1),
    classwise["support"].to_numpy()
)

assert math.isclose(
    metrics["accuracy"],
    173 / 248,
    abs_tol=1e-12
)

evaluation_zip = evaluation_dir / "evaluation_results.zip"
training_zip = root / "training_backup.zip"

for archive_path in (training_zip, evaluation_zip):
    assert archive_path.is_file()

    with zipfile.ZipFile(archive_path) as archive:
        assert archive.testzip() is None

print("D1 CLASS MAPPING")
print(json.dumps(
    class_maps,
    indent=2,
    ensure_ascii=False
))

print("\nTEST RESULT AUDIT")
print("Test samples:", confusion.sum())
print("Correct predictions:", np.trace(confusion))
print("Accuracy:", round(metrics["accuracy"], 6))
print("Class-wise support verified: True")
print("Both result archives verified: True")

print("\nPAPER 6 D1 FINAL EVIDENCE AUDIT: PASSED")

display(FileLink(str(training_zip)))
display(FileLink(str(evaluation_zip)))


D1 CLASS MAPPING
{
  "D1": {
    "Bacterial Leaf Blight": 0,
    "Brown Spot": 1,
    "Healthy Rice Leaf": 2,
    "Leaf Blast": 3,
    "Leaf Scald": 4,
    "Narrow Brown Leaf Spot": 5,
    "Rice Hispa": 6,
    "Sheath Blight": 7
  },
  "D2": {
    "Bacterial Blight": 0,
    "Blast": 1,
    "Brown Spot": 2,
    "Tungro": 3
  }
}

TEST RESULT AUDIT
Test samples: 248
Correct predictions: 173
Accuracy: 0.697581
Class-wise support verified: True
Both result archives verified: True

PAPER 6 D1 FINAL EVIDENCE AUDIT: PASSED


/kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/training_backup.zip

/kaggle/working/Paper6_MFA_SqueezeNet_D1_Revised/evaluation/evaluation_results.zip